In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
import os
# Stops Linux from fragmenting System RAM
os.environ["MALLOC_ARENA_MAX"] = "1"
os.environ["MALLOC_TRIM_THRESHOLD_"] = "10000"

!pip install timm torchmetrics albumentations scipy -q

import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import numpy as np
import cv2
import gc
import math
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torch.optim.swa_utils import AveragedModel, get_ema_multi_avg_fn
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.metrics import cohen_kappa_score
from tqdm.notebook import tqdm

# 👇 FORCED TO SINGLE GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Single GPU Pipeline Ready. Device: {device}")

Single GPU Pipeline Ready. Device: cuda


In [2]:
class Config:
    # Update these paths to your Kaggle dataset
    DATA_DIR = '/kaggle/input/datasets/ascanipek/eyepacs-aptos-messidor-diabetic-retinopathy/augmented_resized_V2' 
    TRAIN_DIR = os.path.join(DATA_DIR, 'train')
    VAL_DIR = os.path.join(DATA_DIR, 'val')
    
    IMG_SIZE = 384
    # 👇 Adjusted for 16GB P100 VRAM. Increase to 20 if it runs fine; drop to 12 if you get CUDA OOM.
    BATCH_SIZE = 12     
    EPOCHS = 3          
    LR = 8e-5            
    NUM_CLASSES = 5
    ORDINAL_CLASSES = 4 
    
    NODE_DIM = 512       
    K_NEIGHBORS = 8      

cfg = Config()

In [3]:
class RetinalGraphDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths, self.labels = [], []
        
        for label in range(cfg.NUM_CLASSES):
            class_dir = os.path.join(root_dir, str(label))
            if os.path.exists(class_dir):
                for img_name in os.listdir(class_dir):
                    self.image_paths.append(os.path.join(class_dir, img_name))
                    self.labels.append(label)

    def __len__(self): return len(self.image_paths)

    def __getitem__(self, idx):
        # PIL Loading is safe from C++ memory leaks
        img_pil = Image.open(self.image_paths[idx]).convert('RGB')
        img_np = np.array(img_pil)
        
        # On-the-go Ben Graham
        img_np = cv2.resize(img_np, (cfg.IMG_SIZE, cfg.IMG_SIZE))
        img_np = cv2.addWeighted(img_np, 4, cv2.GaussianBlur(img_np, (0,0), 10), -4, 128)
        
        if self.transform:
            augmented = self.transform(image=img_np)
            img_tensor = augmented['image']
            
        label = self.labels[idx]
        ordinal_target = torch.zeros(cfg.ORDINAL_CLASSES, dtype=torch.float32)
        if label > 0: ordinal_target[:label] = 1.0
            
        return img_tensor, ordinal_target, torch.tensor(label, dtype=torch.long)

In [4]:
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.5), A.RandomRotate90(p=0.5),
    A.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.05, p=0.4),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

# Use 4 workers for speed; pin_memory=False to save System RAM
train_loader = DataLoader(RetinalGraphDataset(cfg.TRAIN_DIR, train_transform), 
                          batch_size=cfg.BATCH_SIZE, shuffle=True, 
                          num_workers=4, pin_memory=False)

val_loader = DataLoader(RetinalGraphDataset(cfg.VAL_DIR, val_transform), 
                        batch_size=cfg.BATCH_SIZE, shuffle=False, 
                        num_workers=4, pin_memory=False)

In [5]:
class GraphReasoning(nn.Module):
    def __init__(self, dim, k=8):
        super().__init__()
        self.k = k
        self.proj = nn.Linear(dim, dim)
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        B, N, C = x.shape
        dist = torch.cdist(x, x)
        _, indices = torch.topk(dist, k=self.k, largest=False)
        attn = torch.exp(-dist / math.sqrt(C))
        mask = torch.zeros_like(attn).scatter_(-1, indices, 1.0)
        attn = F.softmax(attn.masked_fill(mask == 0, -1e9), dim=-1)
        out = torch.bmm(attn, x)
        return self.norm(x + self.proj(out))

class G_Trans_DSAF(nn.Module):
    def __init__(self):
        super().__init__()
        self.stream_a = timm.create_model('maxvit_tiny_tf_384', pretrained=True, num_classes=0)
        self.stream_b = timm.create_model('resnet18', pretrained=True, num_classes=0, global_pool='')
        self.align_b = nn.Conv2d(512, 512, kernel_size=1) 
        self.graph_reasoning = GraphReasoning(dim=512, k=cfg.K_NEIGHBORS)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Linear(512, cfg.ORDINAL_CLASSES)

    def forward(self, x):
        feat_a = self.stream_a.forward_features(x)
        B, C, H, W = feat_a.shape
        nodes_a = feat_a.view(B, C, H*W).transpose(1, 2)
        
        feat_b = self.stream_b(x)
        feat_b = self.align_b(feat_b)
        nodes_b = feat_b.view(B, C, H*W).transpose(1, 2)
        
        combined_nodes = (nodes_a + nodes_b) / 2
        graph_out = self.graph_reasoning(combined_nodes)
        
        v_a = self.pool(nodes_a.transpose(1, 2)).squeeze(-1)
        v_b = self.pool(nodes_b.transpose(1, 2)).squeeze(-1)
        
        logits = self.classifier(self.pool(graph_out.transpose(1, 2)).squeeze(-1))
        return logits, v_a, v_b

In [6]:
def orthogonal_loss(v_a, v_b):
    return torch.mean(F.cosine_similarity(v_a, v_b, dim=1) ** 2)

class G_DSAFLoss(nn.Module):
    def __init__(self, ortho_weight=0.1):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.ortho_weight = ortho_weight
        
    def forward(self, logits, targets, v_a, v_b):
        l_cls = self.bce(logits, targets)
        l_ortho = orthogonal_loss(v_a, v_b)
        return l_cls + (self.ortho_weight * l_ortho)

# 👇 NO DATAPARALLEL WRAPPER
model = G_Trans_DSAF().to(device)

ema_avg_fn = get_ema_multi_avg_fn(decay=0.999)
ema_model = AveragedModel(model, multi_avg_fn=ema_avg_fn).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.EPOCHS)
criterion = G_DSAFLoss(ortho_weight=0.1)

best_qwk = -1.0

In [ ]:
for epoch in range(cfg.EPOCHS):
    model.train()
    train_loss = 0
    for imgs, targets, _ in tqdm(train_loader, desc=f"Epoch {epoch+1} [Train]"):
        imgs, targets = imgs.to(device), targets.to(device)
        optimizer.zero_grad()
        logits, v_a, v_b = model(imgs)
        loss = criterion(logits, targets, v_a, v_b).mean()
        loss.backward()
        optimizer.step()
        ema_model.update_parameters(model)
        train_loss += loss.item()
        
        del imgs, targets, logits, v_a, v_b, loss
        
    ema_model.eval()
    val_preds, val_trues = [], []
    with torch.no_grad():
        for imgs, targets, labels in tqdm(val_loader, desc=f"Epoch {epoch+1} [Val]"):
            imgs = imgs.to(device)
            logits, _, _ = ema_model(imgs)
            val_preds.extend((torch.sigmoid(logits) > 0.5).sum(dim=1).cpu().numpy())
            val_trues.extend(labels.numpy())
            del imgs, logits
            
    qwk = cohen_kappa_score(val_trues, val_preds, weights='quadratic')
    print(f"Epoch {epoch+1} - QWK: {qwk:.4f}")
    
    if qwk > best_qwk:
        best_qwk = qwk
        torch.save(model.state_dict(), 'g_trans_dsaf_p100_best.pth')
        print("--> Saved New Best Model!")
        
    scheduler.step()
    gc.collect()

Epoch 1 [Train]:   0%|          | 0/9604 [00:00<?, ?it/s]

Epoch 1 [Val]:   0%|          | 0/1186 [00:00<?, ?it/s]

Epoch 1 - QWK: 0.8927
--> Saved New Best Model!


Epoch 2 [Train]:   0%|          | 0/9604 [00:00<?, ?it/s]

Epoch 2 [Val]:   0%|          | 0/1186 [00:00<?, ?it/s]

Epoch 2 - QWK: 0.9010
--> Saved New Best Model!


Epoch 3 [Train]:   0%|          | 0/9604 [00:00<?, ?it/s]